# KaNoN Training

End-to-end recipe for training KaNoN, the conditional-normalizing-flow redshift model, on precomputed spectral embeddings (`KaNoN.py` in this folder holds the model).

**Prerequisites** (paths are placeholders -- update them for your data):

* `merged_filtered_sample.hdf5` -- catalog with `desi_z`, `sdss_z`, `spectype`
* `embeddings.h5` -- per-object 32-dim embeddings under the key `aristotle` (or `plato`)

**Run the cells in order.** Stages 1-3 precompute auxiliary maps that the training stage loads from disk; several cells reuse `phi_clean` / `z_clean` defined in Stage 1-2.

Throughout, the *training mask* keeps only objects whose SDSS and DESI redshifts agree within class-dependent tolerances (GALAXY: 0.33%, QSO: 1%) -- the same "golden sample" filter used by `KaNoNboost_train.py`.

## Stage 1 -- Local manifold scale (`sig_phi`)

For every training object, find the distance to its 50th nearest neighbor in embedding space (FAISS IVF index for speed). This "local scale" sizes the density-aware noise augmentation in `KaNoN.forward_loss`: embeddings in dense regions get small jitter, sparse regions get more.

In [ ]:
import time

import h5py
import numpy as np
import faiss

# --- Hyperparameters ---
K_NEIGHBORS = 50       # distance to the 50th neighbor defines the local scale
N_LIST = 4096          # Voronoi cells (centroids); ~4*sqrt(N) is a good default
N_PROBE = 32           # cells searched per query; higher = more accurate, slower

print(">>> Loading Data...")
f = h5py.File('merged_filtered_sample.hdf5', 'r')
embeddings = h5py.File('embeddings.h5', 'r')['aristotle'][:]
desi_z = f['desi_z'][:]
sdss_z = f['sdss_z'][:]
spectype = f['spectype'][:].astype(str)
f.close()

# --- Class-dependent concordance mask (the "golden sample") ---
print(">>> Applying Class-Dependent Mask...")
delta_z = np.abs(desi_z - sdss_z) / (1 + desi_z)
training_mask = ((spectype == 'GALAXY') & (delta_z < 0.0033)) | \
                ((spectype == 'QSO') & (delta_z < 0.01))

clean_embeddings = embeddings[training_mask].astype('float32')
N, D = clean_embeddings.shape
print(f"    Total objects: {len(embeddings)}")
print(f"    Masked objects (Training Set): {N}")
print(f"    Embedding Dim: {D}")

# --- Build and train the IVF index on this latent space ---
print(f">>> Initializing FAISS IndexIVFFlat (nlist={N_LIST})...")
quantizer = faiss.IndexFlatL2(D)
index = faiss.IndexIVFFlat(quantizer, D, N_LIST, faiss.METRIC_L2)

print(">>> Training Index (Finding Centroids)...")
t0 = time.time()
index.train(clean_embeddings)
print(f"    Training complete in {time.time() - t0:.2f}s")

print(">>> Populating Index...")
index.add(clean_embeddings)

# --- k-NN search of the training set against itself ---
print(f">>> Searching for {K_NEIGHBORS} nearest neighbors (nprobe={N_PROBE})...")
index.nprobe = N_PROBE
t0 = time.time()
D_matrix, I_matrix = index.search(clean_embeddings, K_NEIGHBORS)  # squared L2!
print(f"    Search complete in {time.time() - t0:.2f}s")

# --- Distance to the K-th neighbor (sqrt of the squared distance) ---
print(">>> Extracting density metric (sig_phi)...")
sig_phi = np.sqrt(D_matrix[:, -1])

output_name = 'sig_phi_lookup_fixed.npy'
np.save(output_name, sig_phi)
print(f">>> SUCCESS. Saved {len(sig_phi)} entries to '{output_name}'")
print(f"    Array shape matches training_mask sum: {sig_phi.shape[0] == training_mask.sum()}")

## Stage 2 -- Local redshift gradients (`z_gradients`)

Batched local linear regression: for each object, fit dz/dphi over its 50 nearest neighbors via least squares (GPU `lstsq` when available). *Note: the current `KaNoN.forward_loss` keeps this input in its signature but does not use it -- the tangent-slide augmentation it powered was removed. The map is still produced so precomputed data stays compatible with the loaders below.*

In [ ]:
import h5py
import numpy as np
import faiss
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
K_NEIGHBORS = 50   # neighbors for the local regression
CHUNK_SIZE = 10000

print(">>> Loading Data...")
f = h5py.File('merged_filtered_sample.hdf5', 'r')
embeddings = h5py.File('embeddings.h5', 'r')['aristotle'][:]
desi_z = f['desi_z'][:]
sdss_z = f['sdss_z'][:]
spectype = f['spectype'][:].astype(str)
f.close()

# Exact same mask as Stage 1 so rows stay aligned with sig_phi.
delta_z = np.abs(desi_z - sdss_z) / (1 + desi_z)
training_mask = ((spectype == 'GALAXY') & (delta_z < 0.0033)) | \
                ((spectype == 'QSO') & (delta_z < 0.01))

phi_clean = embeddings[training_mask].astype('float32')
z_clean = desi_z[training_mask].astype('float32')

N, D = phi_clean.shape
print(f"    Data Shape: {N} objects x {D} dimensions")

# --- FAISS index for neighbor lookup ---
print(">>> Building FAISS Index...")
quantizer = faiss.IndexFlatL2(D)
index = faiss.IndexIVFFlat(quantizer, D, 4096, faiss.METRIC_L2)
index.train(phi_clean)
index.add(phi_clean)
index.nprobe = 16

# --- Chunked batch linear regression: solve X @ grad = Y per object ---
print(">>> Calculating Local Gradients (This may take a few minutes)...")
gradients = np.zeros((N, D), dtype='float32')

for i in range(0, N, CHUNK_SIZE):
    end = min(i + CHUNK_SIZE, N)
    batch_phi = phi_clean[i:end]

    D_sq, I = index.search(batch_phi, K_NEIGHBORS)

    neigh_phi = phi_clean[I]   # (Batch, K, D)
    neigh_z = z_clean[I]       # (Batch, K)

    # Center the neighbor cloud on each query point.
    X = neigh_phi - batch_phi[:, None, :]      # dPhi
    Y = neigh_z - z_clean[i:end, None]         # dZ

    X_t = torch.from_numpy(X).to(device)
    Y_t = torch.from_numpy(Y).unsqueeze(-1).to(device)  # (Batch, K, 1)

    try:
        sol = torch.linalg.lstsq(X_t, Y_t).solution     # (Batch, D, 1)
        gradients[i:end] = sol.squeeze(-1).cpu().numpy()
    except Exception as e:
        print(f"Warning: Singular matrix in chunk {i}-{end}. Zeros used.")

    if i % 100000 == 0:
        print(f"    Processed {i}/{N}...")

np.save("z_gradients.npy", gradients)
print(">>> DONE. Saved 'z_gradients.npy'")

In [ ]:
import numpy as np

print("Sanitizing pre-computed maps...")
z_grads = np.load("z_gradients.npy")
sig_phi = np.load("sig_phi_lookup_fixed.npy")

# NaN/Inf can slip through degenerate lstsq solves or empty neighborhoods.
bad_grads = ~np.isfinite(z_grads).all(axis=1)
bad_sigs = ~np.isfinite(sig_phi)
print(f"Found {bad_grads.sum()} objects with bad gradients.")
print(f"Found {bad_sigs.sum()} objects with bad sigmas.")

# Replace: zero gradient (no local slope), mean sigma (typical local scale).
z_grads[bad_grads] = 0.0
sig_phi[bad_sigs] = np.mean(sig_phi[~bad_sigs])

np.save("z_gradients_clean.npy", z_grads)
np.save("sig_phi_clean.npy", sig_phi)
print("Saved clean files. The training stage loads these.")

## Stage 3 -- k-NN redshift anchors

Two k-NN regressors on `(phi_clean, z_clean)` from Stage 2:

* **CV anchors** -- 3-fold out-of-fold predictions, so every training object's anchor comes from a model that never saw it (no leakage). These become the residual targets and the z-binned sigma statistics.
* **Master regressor** -- fit on everything; imported into `KaNoN.anchor` for inference on new objects.

In [ ]:
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_predict

# Uses phi_clean / z_clean from Stage 2.

print(">>> Step 1: Generating CV Anchors (Anti-Leakage)...")
# 3-fold CV: each point's anchor is predicted by a model trained on the other
# 2/3 of the data, simulating the "unknown object" scenario.
temp_knn = KNeighborsRegressor(n_neighbors=10, weights='distance', n_jobs=-1)
z_knn_cv_raw = cross_val_predict(temp_knn, phi_clean, z_clean, cv=3, n_jobs=-1)

np.save("z_knn_cv_raw.npy", z_knn_cv_raw)
print("    Saved 'z_knn_cv_raw.npy'")

print(">>> Step 2: Training Master Regressor...")
# This object is loaded into the KaNoN model for inference.
master_knn = KNeighborsRegressor(n_neighbors=10, weights='distance', n_jobs=-1)
master_knn.fit(phi_clean, z_clean)
joblib.dump(master_knn, "master_knn.pkl")
print("    Saved 'master_knn.pkl'")

## Stage 4 -- Simultaneous decoupled training

The two flows share every forward pass but train independently: separate Adam optimizers, separate early-stopping/LR-annealing state machines keyed to their own validation NLLs (`nll_z` for the physics flow, `nll_phi` for the manifold flow). Each subnetwork cycles `fixed LR -> patience exhausted -> reload best + anneal at the next LR stage -> fixed` until the LR stages run out, then freezes. Training uses a `WeightedRandomSampler` that flattens the redshift distribution so high-z objects are not drowned out.

In [ ]:
%matplotlib inline
import os
from collections import defaultdict

import numpy as np
import h5py
import joblib
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from livelossplot import PlotLosses
import matplotlib.pyplot as plt

from KaNoN import KaNoN

print("""
================================================
KaNoN Simultaneous Decoupled Training
================================================
""")

# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4096
EPOCHS = 500
PATIENCE = 15
BEST_MODEL_PATH = "best_KaNoN.pth"

RESUME_TRAINING = False   # True: load submodule checkpoints below
WARMUP_EPOCHS = 5         # linear LR warmup at the first stage
LR_STAGES = [1e-4, 1e-5, 1e-6]

# --- 1. Data loading (same golden-sample mask as the precompute stages) ---
print("Loading Data...")
f = h5py.File('merged_filtered_sample.hdf5', 'r')
embeddings = h5py.File('embeddings.h5', 'r')['aristotle'][:]
desi_z = f['desi_z'][:]
sdss_z = f['sdss_z'][:]
spectype = f['spectype'][:].astype(str)
f.close()

delta_z = np.abs(desi_z - sdss_z) / (1 + desi_z)
training_mask = ((spectype == 'GALAXY') & (delta_z < 0.0033)) | \
                ((spectype == 'QSO') & (delta_z < 0.01))

embeddings = embeddings[training_mask]
desi_z = desi_z[training_mask]
sdss_z = sdss_z[training_mask]

# Consensus redshift: mean of the two surveys (they already agree to tolerance).
phi_all = embeddings.astype('float32')
z_all = np.mean([desi_z, sdss_z], axis=0).astype('float32')
if z_all.ndim == 1:
    z_all = z_all[:, None]

# --- Auxiliary maps from Stages 1-3 ---
print("Loading Auxiliary Maps...")
sig_phi_all = np.load("sig_phi_clean.npy").astype('float32')
z_grads_raw = np.load("z_gradients_clean.npy").astype('float32')
z_knn_cv = np.load("z_knn_cv_raw.npy").astype('float32')
if z_knn_cv.ndim == 1:
    z_knn_cv = z_knn_cv[:, None]
master_knn = joblib.load("master_knn.pkl")

# --- 2. Dataset & loaders ---
class SpectraDataset(Dataset):
    """Bundles (phi, z, CV anchor, gradient, sig_phi) per object."""
    def __init__(self, phi, z, z_knn, z_grad, sig_phi):
        self.phi = torch.from_numpy(phi).float()
        self.z = torch.from_numpy(z).float()
        self.z_knn = torch.from_numpy(z_knn).float()
        self.z_grad = torch.from_numpy(z_grad).float()
        self.sig_phi = torch.from_numpy(sig_phi).float()

    def __len__(self):
        return len(self.phi)

    def __getitem__(self, idx):
        return self.phi[idx], self.z[idx], self.z_knn[idx], self.z_grad[idx], self.sig_phi[idx]

indices = np.arange(len(embeddings))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_dataset = SpectraDataset(
    phi_all[train_idx], z_all[train_idx], z_knn_cv[train_idx],
    z_grads_raw[train_idx], sig_phi_all[train_idx],
)
val_dataset = SpectraDataset(
    phi_all[val_idx], z_all[val_idx], z_knn_cv[val_idx],
    z_grads_raw[val_idx], sig_phi_all[val_idx],
)

def flatten_redshift(z_data, nbins=200):
    """WeightedRandomSampler with weights ~ 1/bin count: flat N(z) batches."""
    z_data = z_data.flatten()
    counts, bins = np.histogram(z_data, bins=nbins)
    idxs = np.digitize(z_data, bins[:-1]) - 1
    idxs = np.clip(idxs, 0, nbins - 1)
    weights_per_bin = 1.0 / (counts + 1e-8)
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights_per_bin[idxs]).float(),
        num_samples=len(z_data), replacement=True,
    )

print("Creating Sampler...")
sampler = flatten_redshift(z_all[train_idx])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=8, persistent_workers=True, pin_memory=True,
                          drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=8, persistent_workers=True)

# --- 3. Model initialization ---
print("\nInitializing KaNoN...")
init_data = np.hstack([phi_all[train_idx], z_all[train_idx]])
model = KaNoN(
    init_data=init_data,
    pretrained_knn=master_knn,
    precomputed_cv_anchors=z_knn_cv[train_idx],
)
if RESUME_TRAINING:
    print("RESUME FLAG ACTIVE: Looking for submodule checkpoints...")
    z_ckpt = 'best_physics_submodule.pth'
    phi_ckpt = 'best_manifold_submodule.pth'

    if os.path.exists(z_ckpt):
        print(f"  -> Successfully loaded Physics Flow from {z_ckpt}")
        model.physics.load_state_dict(torch.load(z_ckpt))
    else:
        print(f"  -> WARNING: {z_ckpt} not found. Physics Flow starting fresh.")

    if os.path.exists(phi_ckpt):
        print(f"  -> Successfully loaded Manifold Flow from {phi_ckpt}")
        model.manifold_flow.load_state_dict(torch.load(phi_ckpt))
    else:
        print(f"  -> WARNING: {phi_ckpt} not found. Manifold Flow starting fresh.")
else:
    print("STARTING FRESH: Initializing new weights from data stats.")

model.to(device)

# ====================================================================
# --- 4. SIMULTANEOUS DECOUPLED TRAINING ---
# ====================================================================
print(f"\n{'='*50}")
print(" STARTING SIMULTANEOUS DECOUPLED TRAINING")
print(f"{'='*50}")

# Independent optimizers and per-subnetwork state machines.
opt_z = optim.Adam(model.physics.parameters(), lr=LR_STAGES[0])
opt_phi = optim.Adam(model.manifold_flow.parameters(), lr=LR_STAGES[0])

sched_z, sched_phi = None, None

state = {
    'z':   {'phase': 'fixed', 'lr_idx': 0, 'pat': 0, 'best_val': float('inf'),
            'done': False, 'ckpt': 'best_physics_submodule.pth'},
    'phi': {'phase': 'fixed', 'lr_idx': 0, 'pat': 0, 'best_val': float('inf'),
            'done': False, 'ckpt': 'best_manifold_submodule.pth'},
}

liveloss = PlotLosses()

for epoch in range(EPOCHS):
    if state['z']['done'] and state['phi']['done']:
        print("Both networks fully converged. Training complete!")
        break

    # --- Linear warmup (first LR stage only) ---
    if epoch < WARMUP_EPOCHS:
        if not state['z']['done'] and state['z']['phase'] == 'fixed' and state['z']['lr_idx'] == 0:
            for pg in opt_z.param_groups:
                pg['lr'] = LR_STAGES[0] * ((epoch + 1) / WARMUP_EPOCHS)
        if not state['phi']['done'] and state['phi']['phase'] == 'fixed' and state['phi']['lr_idx'] == 0:
            for pg in opt_phi.param_groups:
                pg['lr'] = LR_STAGES[0] * ((epoch + 1) / WARMUP_EPOCHS)

    # --- Train ---
    model.train()
    if state['z']['done']:
        model.physics.eval()
    if state['phi']['done']:
        model.manifold_flow.eval()

    train_metrics = defaultdict(float)
    denom = 0

    for phi, z, z_knn, z_grad, sig_phi in train_loader:
        phi, z, z_knn, z_grad, sig_phi = (phi.to(device), z.to(device), z_knn.to(device),
                                          z_grad.to(device), sig_phi.to(device))
        if sig_phi.ndim == 1:
            sig_phi = sig_phi.unsqueeze(1)

        opt_z.zero_grad()
        opt_phi.zero_grad()

        # One forward computes both NLLs.
        loss, loss_dict = model.forward_loss(phi, z, z_knn, z_grad, sig_phi)

        if torch.isnan(loss):
            opt_z.zero_grad()
            opt_phi.zero_grad()
            continue

        loss.backward()

        # Step each subnetwork independently (skipping frozen ones).
        if not state['z']['done']:
            torch.nn.utils.clip_grad_norm_(model.physics.parameters(), 2.0)
            opt_z.step()
        if not state['phi']['done']:
            torch.nn.utils.clip_grad_norm_(model.manifold_flow.parameters(), 2.0)
            opt_phi.step()

        train_metrics['total_loss'] += loss.item()
        for k, v in loss_dict.items():
            train_metrics[k] += v
        denom += 1

    avg_train = {k: v / denom for k, v in train_metrics.items()}

    # --- Validation (dummy gradient/sigma: neither affects the val NLLs
    #     we track, since jitter/augmentation only run in training mode) ---
    model.eval()
    val_metrics = defaultdict(float)
    val_denom = 0

    with torch.no_grad():
        for phi, z, z_knn_cv_val, _, _ in val_loader:
            phi, z, z_knn_cv_val = phi.to(device), z.to(device), z_knn_cv_val.to(device)
            dummy_grad = torch.zeros_like(phi)
            dummy_sigma = torch.zeros((phi.shape[0], 1), device=device)

            _, loss_dict = model.forward_loss(phi, z, z_knn_cv_val, dummy_grad, dummy_sigma)
            for k, v in loss_dict.items():
                val_metrics[f"val_{k}"] += v
            val_denom += 1

    avg_val = {k: v / val_denom for k, v in val_metrics.items()}

    # --- Logging ---
    liveloss.update({**avg_train, **avg_val})
    liveloss.send()

    # --- Independent early stopping & LR annealing per subnetwork ---
    def process_state(name, val_metric, opt, sched, submodule):
        """fixed -> (patience) -> reload best + cosine anneal -> fixed, until
        the LR stages are exhausted; then freeze the subnetwork."""
        s = state[name]
        if s['done']:
            return opt, sched

        current_val = avg_val[val_metric]
        print(f"[{name.upper()}] Epoch {epoch} - Phase: {s['phase']} | "
              f"Val Metric ({val_metric}): {current_val:.4f}")

        if current_val < s['best_val']:
            s['best_val'] = current_val
            torch.save(submodule.state_dict(), s['ckpt'])
            if s['phase'] == 'fixed':
                s['pat'] = 0
        else:
            if s['phase'] == 'fixed':
                s['pat'] += 1

        if s['phase'] == 'fixed' and s['pat'] >= PATIENCE:
            if s['lr_idx'] >= len(LR_STAGES) - 1:
                print(f"  -> {name.upper()} FLOW FINISHED CONVERGING.")
                s['done'] = True
                # Freeze to save compute on the backward pass.
                for p in submodule.parameters():
                    p.requires_grad = False
                return opt, sched

            print(f"  -> {name.upper()} Patience reached. "
                  f"Annealing LR to {LR_STAGES[s['lr_idx'] + 1]}...")
            submodule.load_state_dict(torch.load(s['ckpt']))
            s['lr_idx'] += 1

            opt = optim.Adam(submodule.parameters(), lr=LR_STAGES[s['lr_idx']])
            sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PATIENCE, eta_min=0.0)
            s['phase'] = 'annealing'
            s['pat'] = 0

        elif s['phase'] == 'annealing':
            sched.step()
            s['pat'] += 1
            if s['pat'] >= PATIENCE:
                print(f"  -> {name.upper()} Annealing done. Resuming fixed LR.")
                s['phase'] = 'fixed'
                s['pat'] = 0

        return opt, sched

    opt_z, sched_z = process_state('z', 'val_nll_z', opt_z, sched_z, model.physics)
    opt_phi, sched_phi = process_state('phi', 'val_nll_phi', opt_phi, sched_phi, model.manifold_flow)

# --- Final assembly: best submodules -> one master checkpoint ---
print("\n--- Assembling Final Master Checkpoint ---")
model.physics.load_state_dict(torch.load(state['z']['ckpt']))
model.manifold_flow.load_state_dict(torch.load(state['phi']['ckpt']))
torch.save(model.state_dict(), BEST_MODEL_PATH)
print(f"Master weights safely compiled and saved to: {BEST_MODEL_PATH}")


# ====================================================================
# --- 5. FINAL EVALUATION & PLOTTING ---
# ====================================================================
print("\n================================================")
print(" Running Final Evaluation on Validation Set...")
print("================================================")

model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

z_true_list = []
z_pred_list = []

with torch.no_grad():
    for phi, z, _, _, _ in val_loader:
        phi = phi.to(device)
        # Coarser grid is fine for a quick end-of-training check.
        z_mode, _ = model.predict_z(phi, n_grid=2000)
        z_true_list.extend(z.cpu().numpy().flatten())
        z_pred_list.extend(z_mode.cpu().numpy().flatten())

z_true = np.array(z_true_list)
z_pred = np.array(z_pred_list)

# Align spectypes using the deterministic val_idx.
spectypes_val = spectype[training_mask][val_idx]
mask_gal = (spectypes_val == 'GALAXY')
mask_qso = (spectypes_val == 'QSO')

def calculate_metrics(zt, zp):
    """R^2, median normalized bias, and sigma_NMAD."""
    r2 = r2_score(zt, zp)
    residuals_norm = (zp - zt) / (1 + zt)
    bias = np.median(residuals_norm)
    sigma_nmad = 1.4826 * np.median(np.abs(residuals_norm - bias))
    return r2, bias, sigma_nmad

g_r2, g_bias, g_nmad = calculate_metrics(z_true, z_pred)
gal_r2, gal_bias, gal_nmad = calculate_metrics(z_true[mask_gal], z_pred[mask_gal])
qso_r2, qso_bias, qso_nmad = calculate_metrics(z_true[mask_qso], z_pred[mask_qso])

print("\n--- GLOBAL METRICS ---")
print(f"R2:         {g_r2:.5f}")
print(f"Bias:       {g_bias:.5f}")
print(f"Sigma NMAD: {g_nmad:.5f}")

print("\n--- GALAXY METRICS ---")
print(f"R2:         {gal_r2:.5f}")
print(f"Bias:       {gal_bias:.5f}")
print(f"Sigma NMAD: {gal_nmad:.5f}")

print("\n--- QSO METRICS ---")
print(f"R2:         {qso_r2:.5f}")
print(f"Bias:       {qso_bias:.5f}")
print(f"Sigma NMAD: {qso_nmad:.5f}")

# --- Hexbin scatter: GALAXY vs QSO ---
fig, ax = plt.subplots(1, 2, figsize=(16, 7), dpi=100)
props = dict(boxstyle='round', facecolor='black', alpha=0.7, edgecolor='none')

hb1 = ax[0].hexbin(z_true[mask_gal], z_pred[mask_gal], gridsize=150,
                   cmap='inferno', bins='log', mincnt=1)
ax[0].plot([0, 2.0], [0, 2.0], 'w--', lw=1.5, label='Perfect Fit')
ax[0].set_xlabel(r"True Redshift ($z_{spec}$)", fontsize=14)
ax[0].set_ylabel(r"Predicted Redshift ($z_{pred}$)", fontsize=14)
ax[0].set_title("KaNoN: GALAXY", fontsize=16)
text_gal = (rf"$R^2$: {gal_r2:.4f}" "\n" rf"Bias: {gal_bias:.5f}" "\n"
            rf"$\sigma_{{NMAD}}$: {gal_nmad:.5f}")
ax[0].text(0.05, 0.95, text_gal, transform=ax[0].transAxes, fontsize=12,
           verticalalignment='top', color='white', bbox=props)
ax[0].legend(loc='lower right', frameon=True, facecolor='black', labelcolor='white')
cb1 = fig.colorbar(hb1, ax=ax[0])
cb1.set_label('Log10(Count)')

hb2 = ax[1].hexbin(z_true[mask_qso], z_pred[mask_qso], gridsize=150,
                   cmap='inferno', bins='log', mincnt=1)
ax[1].plot([0, 6.0], [0, 6.0], 'w--', lw=1.5, label='Perfect Fit')
ax[1].set_xlabel(r"True Redshift ($z_{spec}$)", fontsize=14)
ax[1].set_ylabel(r"Predicted Redshift ($z_{pred}$)", fontsize=14)
ax[1].set_title("KaNoN: QSO", fontsize=16)
text_qso = (rf"$R^2$: {qso_r2:.4f}" "\n" rf"Bias: {qso_bias:.5f}" "\n"
            rf"$\sigma_{{NMAD}}$: {qso_nmad:.5f}")
ax[1].text(0.05, 0.95, text_qso, transform=ax[1].transAxes, fontsize=12,
           verticalalignment='top', color='white', bbox=props)
ax[1].legend(loc='lower right', frameon=True, facecolor='black', labelcolor='white')
cb2 = fig.colorbar(hb2, ax=ax[1])
cb2.set_label('Log10(Count)')

plt.tight_layout()
plt.savefig('KaNoN_Final_Evaluation.png', dpi=300, bbox_inches='tight')
plt.show()

## Stage 5 -- Full-catalog inference

Runs `KaNoN.predict_full_inference` (posterior grid + all density scores in one pass) over **every** object -- no masks -- and writes an HDF5 catalog with: `z_mode`, `z_std`, `z_knn`, the full posterior (`z_grid`, `probs`), and the OOD/consistency scores `log_p_phi`, `log_p_z_given_phi`, `log_p_joint`.

In [ ]:
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from KaNoN import KaNoN

# --- CONFIG ---
BATCH_SIZE = 2048
BEST_MODEL_PATH = "best_KaNoN.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_FILE = "KaNoN_inference.hdf5"
N_GRID = 4000

# --- 1. Load the FULL dataset (no masks -- inference covers everything) ---
print(">>> Loading Full Dataset (Master File)...")
f = h5py.File('merged_filtered_sample.hdf5', 'r')
embeddings = h5py.File('embeddings.h5', 'r')['aristotle'][:]
desi_z = f['desi_z'][:]
sdss_z = f['sdss_z'][:]
f.close()

z_all = np.mean([desi_z, sdss_z], axis=0).astype('float32')
if z_all.ndim == 1:
    z_all = z_all[:, None]

phi_all = embeddings.astype('float32')
N_TOTAL = len(phi_all)
print(f"Total Objects to Process: {N_TOTAL}")

# --- 2. Inference loader ---
class InferenceDataset(Dataset):
    def __init__(self, phi, z):
        self.phi = torch.from_numpy(phi).float()
        self.z = torch.from_numpy(z).float()

    def __len__(self):
        return len(self.phi)

    def __getitem__(self, idx):
        return self.phi[idx], self.z[idx]

full_dataset = InferenceDataset(phi_all, z_all)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# --- 3. Load the trained model (architecture inferred from the checkpoint) ---
print(">>> Loading Model...")
model = KaNoN.from_pretrained(BEST_MODEL_PATH, device=DEVICE)

# The anchor buffers are plain buffers; make sure they're on the GPU too.
if not model.anchor.X_train.is_cuda and DEVICE.type == 'cuda':
    model.anchor.X_train = model.anchor.X_train.to(DEVICE)
    model.anchor.y_train = model.anchor.y_train.to(DEVICE)

# --- 4. Stream posteriors + scores into the HDF5 catalog ---
print(f">>> Creating HDF5: {OUTPUT_FILE} ...")
with h5py.File(OUTPUT_FILE, "w") as f_out:
    d_z = f_out.create_dataset("z_mode", (N_TOTAL,), dtype='f4')
    d_knn = f_out.create_dataset("z_knn", (N_TOTAL,), dtype='f4')
    d_std = f_out.create_dataset("z_std", (N_TOTAL,), dtype='f4')
    # Chunking matters for I/O performance on the big 2D posterior arrays.
    d_grid = f_out.create_dataset("z_grid", (N_TOTAL, N_GRID), dtype='f4',
                                  chunks=(256, N_GRID))
    d_probs = f_out.create_dataset("probs", (N_TOTAL, N_GRID), dtype='f4',
                                   chunks=(256, N_GRID))

    d_log_phi = f_out.create_dataset("log_p_phi", (N_TOTAL,), dtype='f4')
    d_log_z = f_out.create_dataset("log_p_z_given_phi", (N_TOTAL,), dtype='f4')
    d_log_joint = f_out.create_dataset("log_p_joint", (N_TOTAL,), dtype='f4')

    f_out.create_dataset("z_true", data=z_all.flatten())

    start_idx = 0
    with torch.no_grad():
        for i, (phi, _) in enumerate(full_loader):
            phi = phi.to(DEVICE)
            B = phi.shape[0]

            (z_mode, z_std, z_grid, probs, z_knn,
             lp_phi, lp_z, lp_joint) = model.predict_full_inference(phi, n_grid=N_GRID)

            end_idx = start_idx + B
            d_z[start_idx:end_idx] = z_mode.cpu().numpy()
            d_knn[start_idx:end_idx] = z_knn.cpu().numpy()
            d_std[start_idx:end_idx] = z_std.cpu().numpy()
            d_grid[start_idx:end_idx] = z_grid.cpu().numpy()
            d_probs[start_idx:end_idx] = probs.cpu().numpy()

            d_log_phi[start_idx:end_idx] = lp_phi.cpu().numpy()
            d_log_z[start_idx:end_idx] = lp_z.cpu().numpy()
            d_log_joint[start_idx:end_idx] = lp_joint.cpu().numpy()

            start_idx = end_idx
            if i % 5 == 0:
                print(f"   Processed {start_idx}/{N_TOTAL} objects "
                      f"({(start_idx / N_TOTAL) * 100:.1f}%)", end='\r')

print(f"\n\n>>> SUCCESS. Full catalog saved to {OUTPUT_FILE}")

## Stage 6 -- Quick catalog check

Reads the inference catalog back, plots predicted vs. true, and reports outlier rates at several |dz|/(1+z) thresholds plus the class-dependent survey-concordance rate.

In [ ]:
%matplotlib inline
import h5py
import numpy as np
import matplotlib.pyplot as plt

# --- Load predictions + truth ---
with h5py.File('KaNoN_inference.hdf5', 'r') as f:
    z_true = f['z_true'][:]
    z_pred = f['z_mode'][:]

with h5py.File('merged_filtered_sample.hdf5', 'r') as f:
    spectype = np.char.strip(f['spectype'][:].astype(str))

plt.figure(figsize=(6, 6))
plt.scatter(z_true, z_pred, alpha=0.01, s=2)
plt.xlabel('True z')
plt.ylabel('Predicted z (mode)')
plt.title('KaNoN full-catalog inference')
plt.show()

# --- Outlier rates at several thresholds ---
delta_z_norm = np.abs(z_pred - z_true) / (1 + z_true)
thresholds = [0.15, 0.05, 0.02, 0.01]

print(f"{'Threshold':<20} | {'Count':<10} | {'Rate (%)':<10}")
print("-" * 45)
for t in thresholds:
    mask = delta_z_norm > t
    print(f"|dz|/(1+z) > {t:<7} | {np.sum(mask):<10} | {np.mean(mask) * 100:<10.2f}")

# --- Class-dependent survey-concordance outlier rate ---
# "Consistent" = the prediction matches truth within the same tolerance used
# to define the golden training sample (GALAXY 0.33%, QSO 1%).
is_consistent = ((spectype == 'GALAXY') & (delta_z_norm < 0.0033)) | \
                ((spectype == 'QSO') & (delta_z_norm < 0.01))
relevant = (spectype == 'GALAXY') | (spectype == 'QSO')
outliers = ~is_consistent & relevant

print("\n" + "=" * 50)
print(f"GALAXY+QSO Objects:      {relevant.sum()}")
print(f"Consistency Outliers:    {outliers.sum()}")
print(f"Outlier Rate (Logic):    {outliers.sum() / relevant.sum():.2%}")
for obj, tol in [('GALAXY', 0.0033), ('QSO', 0.01)]:
    m = (spectype == obj)
    rate = np.mean(delta_z_norm[m] >= tol)
    print(f"{obj} Outlier Rate:     {rate:.2%} (Thresh < {tol})")
print("=" * 50)